# Chapter 7: Stochastic Programming

**Modern Operations Research with Python**

In this chapter we move from deterministic optimization to **optimization under uncertainty**. Stochastic Programming (SP) explicitly models random parameters (demand, prices, yields, etc.) using probability distributions or discrete scenarios.

We focus on the most practical approach for beginners: **Two-Stage Stochastic Programming** with recourse, implemented via **Sample Average Approximation (SAA)** using PuLP.

## 7.1 Introduction to Two-Stage Stochastic Programming

- **First-stage (here-and-now)** decisions: made before uncertainty is revealed.
- **Second-stage (recourse)** decisions: made after observing the random outcome (scenario).
- Objective: Minimize first-stage cost + expected second-stage cost.

Classic example: **Multi-product Newsvendor** or **Two-Stage Stochastic Inventory**.

In [ ]:
import numpy as np
import pandas as pd
from pulp import *
import matplotlib.pyplot as plt

np.random.seed(42)
print("Libraries imported successfully")

## 7.2 Example: Two-Stage Stochastic Newsvendor (Single Product)

You must decide **order quantity Q** (first stage) before knowing demand D.
After demand realizes, you can sell min(Q, D) and incur shortage or holding costs.

In [ ]:
# Parameters
c = 10.0      # unit ordering cost
p = 25.0      # unit selling price
h = 2.0       # unit holding cost for leftover
b = 15.0      # unit shortage (backorder) penalty

# Generate scenarios (Sample Average Approximation)
n_scenarios = 1000
demand_scenarios = np.random.normal(loc=150, scale=40, size=n_scenarios).clip(0)  # truncated normal
prob = np.ones(n_scenarios) / n_scenarios

print(f"Generated {n_scenarios} demand scenarios. Mean demand: {demand_scenarios.mean():.1f}")

In [ ]:
# Build the extensive form (deterministic equivalent)
prob = LpProblem("Two_Stage_Newsvendor", LpMinimize)

# First-stage decision
Q = LpVariable("Order_Quantity", lowBound=0, cat="Continuous")

# Second-stage variables (one per scenario)
sales = LpVariable.dicts("Sales", range(n_scenarios), lowBound=0)
leftover = LpVariable.dicts("Leftover", range(n_scenarios), lowBound=0)
shortage = LpVariable.dicts("Shortage", range(n_scenarios), lowBound=0)

# Objective: first-stage cost + expected recourse cost
prob += c * Q + lpSum([prob[i] * (h * leftover[i] + b * shortage[i] - (p - c) * sales[i]) for i in range(n_scenarios)])

# Constraints for each scenario
for i in range(n_scenarios):
    d = demand_scenarios[i]
    prob += sales[i] <= Q
    prob += sales[i] <= d
    prob += leftover[i] == Q - sales[i]
    prob += shortage[i] == d - sales[i]

prob.solve(PULP_CBC_CMD(msg=0))

print("Optimal first-stage order quantity Q:", value(Q))
print("Expected total cost:", value(prob.objective))

## 7.3 Visualization of Solution

In [ ]:
plt.figure(figsize=(10, 6))
plt.hist(demand_scenarios, bins=50, alpha=0.7, label='Demand distribution')
plt.axvline(value(Q), color='red', linestyle='--', linewidth=2, label=f'Optimal Q = {value(Q):.1f}')
plt.xlabel('Demand')
plt.ylabel('Frequency')
plt.title('Two-Stage Stochastic Newsvendor - Optimal Order Quantity')
plt.legend()
plt.show()

## 7.4 Multi-Product Extension (Exercise-ready)

Extend the model to multiple products with shared capacity or budget constraint in the first stage.

Key idea: First-stage decides production/order quantities for all products; second-stage handles sales/shortage per scenario.

In [ ]:
# Placeholder for multi-product version (students can implement)
print("Multi-product stochastic model can be built similarly with indexed variables.")

## 7.5 Key Takeaways & Next Steps

- Stochastic programming captures uncertainty explicitly.
- Sample Average Approximation turns the problem into a large deterministic LP/MIP.
- For very large scenario sets, use decomposition methods (Benders, L-shaped) or specialized solvers.

**Exercises:**
1. Add a capacity constraint on total first-stage order.
2. Compare SAA solution quality as number of scenarios increases.
3. Implement a simple chance-constrained model (e.g., Prob(stockout) ≤ α).

In the next chapter we will explore **Robust Optimization** and **Chance-Constrained Programming** as alternatives to stochastic programming.